# Applicability Domain (AD) Assessment – Cardiosim-Tox

- **Fingerprint-based AD**: Tanimoto similarity on **ECFP4 + MACCS concatenated** (1191-bit), single FP score
- **Leverage-based AD**: Williams approach (RDKit + CDK physicochemical descriptors)
- **Consensus rule**: compound is in-domain if **BOTH** FP_Concat AND Leverage pass
- **τ selection**: highest τ where avg consensus coverage ≥ 70% on EV1+EV2+EV3; fallback τ = 0.5 if none qualify
- **Output**: `D:\QSAR Cardiotoxicity\AD Analysis` – all plots as PNG, results as Excel

In [ ]:
import os
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import DataStructs

warnings.filterwarnings("ignore")
print("Libraries loaded.")

## 1. Paths & Configuration

In [ ]:
FP_ROOT  = r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\Fingerprint"
PP_ROOT  = r"D:\QSAR Cardiotoxicity\Binary_Descriptor Calculation\PP"
OUT_DIR  = r"D:\QSAR Cardiotoxicity\AD Analysis"
os.makedirs(OUT_DIR, exist_ok=True)

TARGETS   = ["Cav1.2", "hERG", "Nav1.5"]
ALL_SETS  = ["DevSet", "ExternalVal1_easy", "ExternalVal2_medium", "ExternalVal3_hard"]
SENS_SETS = ["ExternalVal1_easy", "ExternalVal2_medium", "ExternalVal3_hard"]

TAU_CANDIDATES = [round(x, 2) for x in np.arange(0.30, 0.75, 0.05)]
MIN_COVERAGE   = 70.0    # minimum consensus coverage (%)
FALLBACK_TAU   = 0.50    # used when no τ achieves MIN_COVERAGE

print(f"Output directory : {OUT_DIR}")
print(f"Targets          : {TARGETS}")
print(f"All sets         : {ALL_SETS}")
print(f"Sensitivity sets : {SENS_SETS}")
print(f"Tau candidates   : {TAU_CANDIDATES}")
print(f"Min coverage     : {MIN_COVERAGE}%")
print(f"Fallback tau     : {FALLBACK_TAU}")

## 2. Helper Functions

In [ ]:
# ── Fingerprint parsing ────────────────────────────────────────────────────
def parse_fp_col(series):
    result = []
    for val in series:
        if isinstance(val, (list, np.ndarray)):
            result.append(list(val))
        elif isinstance(val, str):
            result.append(ast.literal_eval(val))
        else:
            result.append([])
    return result

def to_bitvect(fp_list, n_bits=None):
    if not fp_list:
        fp_list = [0] * (n_bits or 1024)
    if n_bits is None:
        n_bits = len(fp_list)
    bv = DataStructs.ExplicitBitVect(n_bits)
    for i, bit in enumerate(fp_list):
        if bit:
            bv.SetBit(i)
    return bv

def concat_fps(ecfp4_list, maccs_list):
    return [e + m for e, m in zip(ecfp4_list, maccs_list)]

def compute_max_tanimoto(test_fps, train_fps):
    train_bvs = [to_bitvect(fp) for fp in train_fps]
    n_bits    = len(train_bvs[0])
    max_sims  = []
    for fp in test_fps:
        bv   = to_bitvect(fp, n_bits=n_bits)
        sims = DataStructs.BulkTanimotoSimilarity(bv, train_bvs)
        max_sims.append(max(sims) if sims else 0.0)
    return np.array(max_sims, dtype=np.float32)

def compute_max_tanimoto_loo(fps):
    bvs      = [to_bitvect(fp) for fp in fps]
    max_sims = []
    for i, bv in enumerate(bvs):
        others = bvs[:i] + bvs[i+1:]
        sims   = DataStructs.BulkTanimotoSimilarity(bv, others)
        max_sims.append(max(sims) if sims else 0.0)
    return np.array(max_sims, dtype=np.float32)

# ── File loaders ───────────────────────────────────────────────────────────
def load_fp_file(path):
    df    = pd.read_csv(path)
    ecfp4 = parse_fp_col(df["ECFP4"])
    maccs = parse_fp_col(df["MACCS"])
    meta  = df[["SMILES"]].copy()
    if "Activity" in df.columns:
        meta["Activity"] = df["Activity"].values
    return meta, ecfp4, maccs

# Whitelist: hanya kolom physicochemical descriptor murni
PP_DESCRIPTOR_COLS = [
    'MolWt','logP','LabuteASA','TPSA','AMW','NumRotatableBonds',
    'NumAromaticRings','NumSaturatedRings','NumAliphaticRings',
    'NumAromaticHeterocycles','NumSaturatedHeterocycles','NumAliphaticHeterocycles',
    'NumAromaticCarbocycles','NumSaturatedCarbocycles','NumAliphaticCarbocycles',
    'FractionCSP3','Chi0v','Chi1v','Chi2v','Chi3v','Chi4v',
    'Chi1n','Chi2n','Chi3n','Chi4n','HallKierAlpha','HeavyAtomCount',
    'RingCount','NumHDonors','NumHAcceptors','ALogP','ALogp2','AMR',
    'MLogP','nAtomP','naAromAtom','bpol','nB','ECCEN','fragC',
    'nHBAcc','nHBDon','nAtomLAC','nAtomLC','PetitjeanNumber','nRotB',
    'LipinskiFailures','TopoPSA','VAdjMat','XLogP','Fsp3'
]

def load_pp_file(path):
    """Return (meta_df, descriptor_df) — whitelist only, no leakage columns."""
    df   = pd.read_excel(path)
    meta = df[["SMILES"]].copy() if "SMILES" in df.columns else pd.DataFrame(index=df.index)
    if "Activity" in df.columns:
        meta["Activity"] = df["Activity"].values
    cols = [c for c in PP_DESCRIPTOR_COLS if c in df.columns]
    return meta, df[cols]

# ── Leverage ───────────────────────────────────────────────────────────────
def compute_leverage_ad(X_train, X_test):
    """
    Williams leverage approach.
    Returns (H_test, h_star, in_domain_bool_array)
    """
    n, p     = X_train.shape
    Xa_train = np.hstack([np.ones((n, 1)), X_train])
    Xa_test  = np.hstack([np.ones((X_test.shape[0], 1)), X_test])
    XtX_inv  = np.linalg.pinv(Xa_train.T @ Xa_train)
    H_test   = np.sum((Xa_test @ XtX_inv) * Xa_test, axis=1)
    h_star   = 3 * (p + 1) / n
    return H_test, h_star, (H_test <= h_star)

print("Helper functions defined.")
print(f"PP descriptor whitelist: {len(PP_DESCRIPTOR_COLS)} columns")

## 3. Main AD Loop

In [ ]:
all_ad_rows   = []
all_sens_rows = []
tau_selected  = {}

for target in TARGETS:
    print(f"\n{'='*60}")
    print(f"  TARGET: {target}")
    print(f"{'='*60}")
    target_dir = os.path.join(OUT_DIR, target)
    os.makedirs(target_dir, exist_ok=True)

    # ── Load DevSet (training reference) ──────────────────────────────────
    fp_dev = os.path.join(FP_ROOT, f"{target}_DevSet_with_ECFP4_MACCS_APF.csv")
    pp_dev = os.path.join(PP_ROOT, f"{target}_DevSet_RDKit_CDK.xlsx")
    if not os.path.exists(fp_dev):
        print(f"  [ERROR] Not found: {fp_dev}"); continue
    if not os.path.exists(pp_dev):
        print(f"  [ERROR] Not found: {pp_dev}"); continue

    _, train_ecfp4, train_maccs = load_fp_file(fp_dev)
    train_concat = concat_fps(train_ecfp4, train_maccs)   # 1191-bit

    # PP: whitelist columns only, fillna with training mean
    _, X_train_df = load_pp_file(pp_dev)
    col_means     = X_train_df.mean()
    X_train_df    = X_train_df.fillna(col_means)
    pp_cols       = X_train_df.columns.tolist()
    X_train       = X_train_df.values.astype(np.float64)
    n_pp, p_pp    = X_train.shape
    h_star        = 3 * (p_pp + 1) / n_pp

    print(f"  Training FP : {len(train_concat)} compounds | concat bits = {len(train_concat[0])}")
    print(f"  Training PP : n={n_pp}, p={p_pp} descriptors, h* = {h_star:.4f}")

    # ── Compute similarity + leverage for every set ────────────────────────
    per_set = {}
    for split in ALL_SETS:
        fp_path = os.path.join(FP_ROOT, f"{target}_{split}_with_ECFP4_MACCS_APF.csv")
        pp_path = os.path.join(PP_ROOT, f"{target}_{split}_RDKit_CDK.xlsx")

        if not os.path.exists(fp_path):
            print(f"  [SKIP FP] {fp_path}"); continue

        meta_q, test_ecfp4, test_maccs = load_fp_file(fp_path)
        test_concat = concat_fps(test_ecfp4, test_maccs)
        n_q         = len(test_concat)

        # Tanimoto on concatenated FP
        if split == "DevSet":
            print(f"  {split}: LOO Tanimoto on concat FP ...")
            sims_fp = compute_max_tanimoto_loo(train_concat)
        else:
            print(f"  {split}: Tanimoto on concat FP ...")
            sims_fp = compute_max_tanimoto(test_concat, train_concat)

        # Leverage — whitelist columns, reindex to training columns
        if os.path.exists(pp_path):
            _, X_q_df  = load_pp_file(pp_path)
            X_q_df     = X_q_df.reindex(columns=pp_cols).fillna(col_means)
            X_q        = X_q_df.values.astype(np.float64)
            H_lev, h_star_q, lev_in = compute_leverage_ad(X_train, X_q)
        else:
            print(f"  [SKIP PP] {pp_path}")
            H_lev    = np.full(n_q, np.nan)
            h_star_q = h_star
            lev_in   = np.zeros(n_q, dtype=bool)

        per_set[split] = dict(
            meta=meta_q, sims_fp=sims_fp,
            H_lev=H_lev, h_star=h_star_q, lev_in=lev_in
        )
        print(f"    FP sim : mean={sims_fp.mean():.3f}, min={sims_fp.min():.3f}, max={sims_fp.max():.3f}"
              f" | Leverage in-domain: {lev_in.mean()*100:.1f}%")

    # ── Build sensitivity table ────────────────────────────────────────────
    for split in ALL_SETS:
        if split not in per_set:
            continue
        d = per_set[split]
        n_total = len(d["sims_fp"])
        for tau in TAU_CANDIDATES:
            fp_in  = d["sims_fp"] >= tau
            lev_in = d["lev_in"]
            cons   = fp_in & lev_in
            all_sens_rows.append({
                "Target"               : target,
                "Split"                : split,
                "Tau"                  : tau,
                "N_total"              : n_total,
                "Coverage_FP_%"        : round(fp_in.mean()  * 100, 2),
                "Coverage_Leverage_%"  : round(lev_in.mean() * 100, 2),
                "Coverage_Consensus_%" : round(cons.mean()   * 100, 2),
            })

    df_sens_t = pd.DataFrame(
        [r for r in all_sens_rows if r["Target"] == target])

    # ── Select tau ────────────────────────────────────────────────────────
    eval_mask = df_sens_t["Split"].isin(SENS_SETS)
    avg_cov   = (df_sens_t[eval_mask]
                 .groupby("Tau")["Coverage_Consensus_%"]
                 .mean().reset_index().sort_values("Tau"))
    valid    = avg_cov[avg_cov["Coverage_Consensus_%"] >= MIN_COVERAGE]

    if valid.empty:
        tau_fp   = FALLBACK_TAU
        fallback = True
        fb_row   = avg_cov[avg_cov["Tau"] == FALLBACK_TAU]
        sel_cov  = fb_row["Coverage_Consensus_%"].values[0] if not fb_row.empty else np.nan
        print(f"  [WARN] No τ achieves {MIN_COVERAGE}% → fallback τ={tau_fp:.2f} "
              f"(coverage={sel_cov:.1f}%)")
    else:
        tau_fp   = float(valid["Tau"].max())
        fallback = False
        sel_cov  = avg_cov.loc[avg_cov["Tau"] == round(tau_fp, 2),
                               "Coverage_Consensus_%"].values[0]
        print(f"  ★ Selected τ_FP = {tau_fp:.2f}  "
              f"(avg consensus coverage = {sel_cov:.1f}%)")

    tau_selected[target] = {"tau": tau_fp, "fallback": fallback, "consensus_cov": sel_cov}

    with open(os.path.join(target_dir, f"{target}_selected_tau.txt"), "w") as f:
        f.write(f"Target                         : {target}\n")
        f.write(f"Selected tau_FP                : {tau_fp:.2f}\n")
        f.write(f"Fallback used                  : {fallback}\n")
        f.write(f"Min consensus coverage required: {MIN_COVERAGE}%\n")
        f.write(f"Avg consensus cov at selected τ: {sel_cov:.2f}%\n")
        f.write(f"FP vector                      : ECFP4 (1024-bit) + MACCS (167-bit) = 1191-bit concat\n")
        f.write(f"PP descriptors used            : {len(pp_cols)} columns\n")
        f.write(f"PP descriptor list             : {pp_cols}\n")

    # ── Apply tau_fp → AD flags for ALL sets ──────────────────────────────
    for split in ALL_SETS:
        if split not in per_set:
            continue
        d      = per_set[split]
        meta_q = d["meta"]
        n_q    = len(d["sims_fp"])

        fp_in  = (d["sims_fp"] >= tau_fp).astype(int)
        lev_in = d["lev_in"].astype(int)
        cons   = (fp_in & lev_in).astype(int)

        print(f"  {split:30s} @ τ={tau_fp:.2f}: "
              f"FP={fp_in.mean()*100:.1f}%  "
              f"Leverage={lev_in.mean()*100:.1f}%  "
              f"Consensus={cons.mean()*100:.1f}%")

        for i in range(n_q):
            smiles   = meta_q["SMILES"].iloc[i]   if "SMILES"   in meta_q.columns else ""
            activity = meta_q["Activity"].iloc[i] if "Activity" in meta_q.columns else np.nan
            h_val    = d["H_lev"][i]
            all_ad_rows.append({
                "Target"            : target,
                "Split"             : split,
                "SMILES"            : smiles,
                "Activity"          : activity,
                "Sim_FP_Concat"     : round(float(d["sims_fp"][i]), 4),
                "Tau_FP"            : tau_fp,
                "FP_InDomain"       : int(fp_in[i]),
                "Leverage_h"        : round(float(h_val), 6) if not np.isnan(h_val) else np.nan,
                "h_star"            : round(d["h_star"], 6),
                "Lev_InDomain"      : int(lev_in[i]),
                "Consensus_InDomain": int(cons[i]),
            })

    print(f"  ✅ {target} done.")

df_ad   = pd.DataFrame(all_ad_rows)
df_sens = pd.DataFrame(all_sens_rows)
print("\n✅ AD computation complete.")

## 4. Save AD Results to Excel

In [ ]:
# ── Coverage summary ───────────────────────────────────────────────────────
summary_rows = []
for target in TARGETS:
    tau_info = tau_selected.get(target, {})
    for split in ALL_SETS:
        sub = df_ad[(df_ad["Target"] == target) & (df_ad["Split"] == split)]
        if sub.empty:
            continue
        n = len(sub)
        summary_rows.append({
            "Target"               : target,
            "Split"                : split,
            "N_total"              : n,
            "Coverage_FP_%"        : round(sub["FP_InDomain"].sum()          / n * 100, 2),
            "Coverage_Leverage_%"  : round(sub["Lev_InDomain"].sum()         / n * 100, 2),
            "Coverage_Consensus_%" : round(sub["Consensus_InDomain"].sum()   / n * 100, 2),
            "Tau_FP_selected"      : sub["Tau_FP"].iloc[0],
            "Fallback_used"        : tau_info.get("fallback", ""),
            "h_star"               : sub["h_star"].iloc[0],
        })
df_summary = pd.DataFrame(summary_rows)

# ── Write AD Excel ─────────────────────────────────────────────────────────
ad_xlsx = os.path.join(OUT_DIR, "AD_Analysis_Results.xlsx")
with pd.ExcelWriter(ad_xlsx, engine="openpyxl") as writer:
    df_ad.to_excel(writer,      sheet_name="AD_Detail",           index=False)
    df_summary.to_excel(writer, sheet_name="AD_Coverage_Summary", index=False)
    df_ad[df_ad["Consensus_InDomain"] == 0].to_excel(
        writer, sheet_name="OutOfDomain_Compounds", index=False)

# ── Write Sensitivity Excel ────────────────────────────────────────────────
sens_xlsx = os.path.join(OUT_DIR, "Sensitivity_Test_Results.xlsx")
with pd.ExcelWriter(sens_xlsx, engine="openpyxl") as writer:
    for target in TARGETS:
        sub = df_sens[df_sens["Target"] == target]
        if not sub.empty:
            sub.to_excel(writer, sheet_name=f"{target}_SensTest", index=False)

print(f"✅ AD Analysis Excel  → {ad_xlsx}")
print(f"✅ Sensitivity Excel  → {sens_xlsx}")
print(f"\nTotal compounds      : {len(df_ad)}")
print(f"Out-of-domain (cons) : {(df_ad['Consensus_InDomain']==0).sum()}")
df_summary

## 5. Sensitivity Plots – per Modality (PNG per target × split)

In [ ]:
MODALITY_PLOT = [
    ("FP (ECFP4+MACCS concat)", "Coverage_FP_%",        "steelblue",  "--", 1.6),
    ("Leverage",                 "Coverage_Leverage_%",  "darkorange", "--", 1.6),
    ("Consensus (FP & Lev)",     "Coverage_Consensus_%", "crimson",    "-",  2.4),
]

for target in TARGETS:
    target_dir = os.path.join(OUT_DIR, target)
    tau_fp     = tau_selected[target]["tau"]
    fallback   = tau_selected[target]["fallback"]
    sub_t      = df_sens[df_sens["Target"] == target]

    for split in ALL_SETS:
        sub = sub_t[sub_t["Split"] == split].sort_values("Tau")
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(7.5, 4.5))

        for (label, col, color, ls, lw) in MODALITY_PLOT:
            ax.plot(sub["Tau"], sub[col],
                    marker="o", color=color, linestyle=ls,
                    linewidth=lw, markersize=5, label=label)

        ax.axhline(MIN_COVERAGE, color="black", linestyle=":",
                   linewidth=1.3, label=f"Min coverage ({MIN_COVERAGE}%)")

        tau_color = "darkorange" if fallback else "red"
        tau_label = (f"Fallback τ = {tau_fp:.2f}" if fallback
                     else f"Selected τ = {tau_fp:.2f}")
        ax.axvline(tau_fp, color=tau_color, linestyle="-.",
                   linewidth=1.8, label=tau_label)

        # annotate consensus coverage at selected tau
        sel_row = sub[sub["Tau"] == round(tau_fp, 2)]
        if not sel_row.empty:
            y_val = sel_row["Coverage_Consensus_%"].values[0]
            ax.scatter([tau_fp], [y_val], s=100, zorder=5, color=tau_color)
            ax.annotate(f"{y_val:.1f}%",
                        xy=(tau_fp, y_val),
                        xytext=(tau_fp + 0.02, y_val + 3),
                        fontsize=8.5, color=tau_color, fontweight="bold")

        split_label = split.replace("ExternalVal", "EV")
        ax.set_xlabel("Tanimoto Threshold (τ)", fontsize=11)
        ax.set_ylabel("AD Coverage (%)", fontsize=11)
        ax.set_title(f"{target} – {split_label}\n"
                     f"Sensitivity Analysis (FP: ECFP4+MACCS concat)",
                     fontsize=11, fontweight="bold")
        ax.set_ylim(0, 110)
        ax.set_xticks(TAU_CANDIDATES)
        ax.legend(fontsize=9, loc="lower left", framealpha=0.85)
        ax.grid(alpha=0.25)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()

        out_png = os.path.join(target_dir,
                               f"{target}_{split}_sensitivity.png")
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"  Saved: {out_png}")

print("\n✅ All sensitivity plots saved.")

## 6. Coverage Bar Charts – per Modality (PNG per target)

In [ ]:
BAR_MODALITIES = [
    ("FP (ECFP4+MACCS)", "Coverage_FP_%",        "steelblue"),
    ("Leverage",          "Coverage_Leverage_%",  "darkorange"),
    ("Consensus",         "Coverage_Consensus_%", "crimson"),
]

for target in TARGETS:
    target_dir = os.path.join(OUT_DIR, target)
    tau_fp     = tau_selected[target]["tau"]
    fallback   = tau_selected[target]["fallback"]
    sub        = df_summary[df_summary["Target"] == target].copy()
    if sub.empty:
        continue

    x_labels = [s.replace("ExternalVal", "EV") for s in sub["Split"].tolist()]
    n_sets   = len(x_labels)
    n_mod    = len(BAR_MODALITIES)
    wid      = 0.22
    x        = np.arange(n_sets)
    offsets  = np.linspace(-(n_mod-1)/2*wid, (n_mod-1)/2*wid, n_mod)

    fig, ax = plt.subplots(figsize=(10, 5))
    for (label, col, color), offset in zip(BAR_MODALITIES, offsets):
        vals = sub[col].tolist()
        bars = ax.bar(x + offset, vals, wid, label=label,
                      color=color, edgecolor="k", linewidth=0.4)
        for rect, v in zip(bars, vals):
            if pd.notna(v):
                ax.text(rect.get_x() + rect.get_width()/2,
                        rect.get_height() + 0.7,
                        f"{v:.1f}%", ha="center", va="bottom",
                        fontsize=7.5, rotation=90)

    ax.axhline(MIN_COVERAGE, color="black", linestyle="--",
               linewidth=1.2, label=f"Min coverage ({MIN_COVERAGE}%)")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_ylabel("AD Coverage (%)", fontsize=11)

    fb_note = " [fallback τ]" if fallback else ""
    ax.set_title(f"{target} – AD Coverage per Modality\n"
                 f"(τ = {tau_fp:.2f}{fb_note} | FP: ECFP4+MACCS concat)",
                 fontsize=12, fontweight="bold")
    ax.set_ylim(0, 125)
    ax.legend(fontsize=9, loc="upper right", framealpha=0.85)
    ax.grid(axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()

    out_png = os.path.join(target_dir, f"{target}_AD_coverage_per_modality.png")
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  Saved: {out_png}")

print("\n✅ Coverage bar charts saved.")

## 7. Overview Plot – Consensus Coverage, All Targets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, target in zip(axes, TARGETS):
    sub      = df_summary[df_summary["Target"] == target]
    if sub.empty:
        continue
    tau_fp   = tau_selected[target]["tau"]
    fallback = tau_selected[target]["fallback"]
    x_labels = [s.replace("ExternalVal", "EV") for s in sub["Split"].tolist()]
    vals     = sub["Coverage_Consensus_%"].tolist()
    colors   = ["tomato" if v < MIN_COVERAGE else "steelblue" for v in vals]

    bars = ax.bar(x_labels, vals, color=colors, edgecolor="k", linewidth=0.6)
    ax.axhline(MIN_COVERAGE, color="red", linestyle="--", linewidth=1.3)
    fb_note = " [FB]" if fallback else ""
    ax.set_title(f"{target}\n(τ = {tau_fp:.2f}{fb_note})",
                 fontsize=12, fontweight="bold")
    ax.set_ylim(0, 115)
    ax.set_ylabel("Consensus AD Coverage (%)" if target == TARGETS[0] else "",
                  fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for rect, v in zip(bars, vals):
        ax.text(rect.get_x() + rect.get_width()/2,
                rect.get_height() + 1.5,
                f"{v:.1f}%", ha="center", fontsize=9)

fig.suptitle(
    "Global Consensus AD Coverage – All Targets & Sets\n"
    "(FP: ECFP4+MACCS concat + Leverage | [FB] = fallback τ used)",
    fontsize=12, y=1.03
)
fig.tight_layout()

out_png = os.path.join(OUT_DIR, "AD_Coverage_Overview.png")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"✅ Overview plot saved → {out_png}")
print(f"\n{'='*60}")
print(f"  ALL DONE – outputs in: {OUT_DIR}")
print(f"{'='*60}")

# Print final tau summary
print("\nSelected τ summary:")
for t, info in tau_selected.items():
    fb = " ← FALLBACK" if info["fallback"] else ""
    print(f"  {t:8s}: τ = {info['tau']:.2f}  "
          f"consensus cov = {info['consensus_cov']:.1f}%{fb}")

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ======================================================
# NEW DATA INPUT
# ======================================================
data = {
    "Endpoint / Dataset": [
        "Cav1.2\nExternalVal1_easy",
        "Cav1.2\nExternalVal2_medium",
        "Cav1.2\nExternalVal3_hard",
        "hERG\nExternalVal1_easy",
        "hERG\nExternalVal2_medium",
        "hERG\nExternalVal3_hard",
        "Nav1.5\nExternalVal1_easy",
        "Nav1.5\nExternalVal2_medium",
        "Nav1.5\nExternalVal3_hard",
    ],
    "% FP-Graph IN AD": [
        100, 100, 50,
        100, 100, 45.85,
        100, 99.6, 60.18
    ],
    "% Mordred IN AD": [
        98.63, 95.59, 93.18,
        99.29, 98.8, 95.33,
        99.6, 97.89, 94.69
    ],
    "% Consensus IN AD": [
        98.63, 93.18, 50,
        99.29, 98.8, 44.63,
        99.2, 97.89, 56.64
    ],
    "Threshold (Tanimoto)": [
        0.7, 0.7, 0.7,
        0.7, 0.7, 0.7,
        0.7, 0.7, 0.7
    ],
}

df = pd.DataFrame(data)

# ======================================================
# HEATMAP DATA
# ======================================================
heatmap_data = df.set_index("Endpoint / Dataset")[
    ["% FP-Graph IN AD", "% Mordred IN AD", "% Consensus IN AD"]
]

fig, (ax_hm, ax_thr) = plt.subplots(
    ncols=2, figsize=(14, 6), gridspec_kw={"width_ratios": [4, 1]}
)

# ======================================================
# 1) HEATMAP
# ======================================================
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    ax=ax_hm,
    cbar_kws={"label": "Coverage (%)"},
)

ax_hm.set_xlabel("Descriptor / Consensus IN AD")
ax_hm.set_ylabel("")
ax_hm.set_title("Applicability Domain Coverage Across Ion Channel Targets")

ymin, ymax = ax_hm.get_ylim()

# ======================================================
# 2) THRESHOLD BARPLOT
# ======================================================
thresholds = df["Threshold (Tanimoto)"].values
y_centers = np.arange(heatmap_data.shape[0]) + 0.5

thr_min, thr_max = 0.5, 0.8
norm = plt.Normalize(vmin=thr_min, vmax=thr_max)
cmap = plt.cm.Reds
colors = cmap(norm(thresholds))

ax_thr.barh(
    y=y_centers,
    width=thresholds,
    color=colors,
    edgecolor="black",
)

ax_thr.set_ylim(ymin, ymax)
ax_thr.set_yticks([])
ax_thr.set_xlabel("Tanimoto\nthreshold")
ax_thr.set_xlim(thr_min, thr_max)

for y, thr in zip(y_centers, thresholds):
    ax_thr.text(
        thr + 0.005,
        y,
        f"{thr:.2f}",
        va="center",
        ha="left",
        fontsize=9,
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax_thr, fraction=0.2, pad=0.2)
cbar.set_label("Tanimoto threshold")

plt.tight_layout()

output_path = r"D:\QSAR Cardiotoxicity\AD Analysis"
os.makedirs(output_path, exist_ok=True)

plt.savefig(
    os.path.join(output_path, "AD_coverage_ion_channel.png"),
    dpi=600,
)
